In [2]:
pip install torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\avina\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
# importing the text for tokenization
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path="the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x1742b3fa990>)

In [4]:
# opening the text file and reading the content
with open(file_path,"r", encoding="utf-8") as f:
    raw_text = f.read()
print("total characters in the text:", len(raw_text))
print(raw_text[:500])

total characters in the text: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it'


In [5]:
# splitting the text into words
import re
text="Hello, how are you doing? I hope you're doing well."
result=re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'how', ' ', 'are', ' ', 'you', ' ', 'doing?', ' ', 'I', ' ', 'hope', ' ', "you're", ' ', 'doing', ' ', 'well', '.', '']


In [6]:
# stripping the whitespace 
result=[item.strip() for item in result if item.strip()]
result

['Hello',
 ',',
 'how',
 'are',
 'you',
 'doing?',
 'I',
 'hope',
 "you're",
 'doing',
 'well',
 '.']

In [7]:
result=re.split(r'([,.:;?_!"()\']|--|\s)', text)
result=[item.strip() for item in result if item.strip()]
result

['Hello',
 ',',
 'how',
 'are',
 'you',
 'doing',
 '?',
 'I',
 'hope',
 'you',
 "'",
 're',
 'doing',
 'well',
 '.']

In [8]:
# converting raw text into a list of words
preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed=[item.strip() for item in preprocessed if item.strip()]
print("total words in the text:", len(preprocessed))


total words in the text: 4690


In [9]:
print(preprocessed[:20])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']


In [10]:
# creating a set of unique words (vocabulary)
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
print("vocabulary size:", vocab_size)

vocabulary size: 1130


In [11]:
# creating vocab
vocab={token:idx for idx, token in enumerate(all_words)}
# printing the first 20 items in the vocab
for idx, token in enumerate(vocab.items()):
    if idx<20:
        print(token)

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)


In [12]:
# creating a simppletokenizer class to convert text into tokens and vice versa
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}
    def encode(self,text):
        preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        ids=[self.str_to_int[s] for s in preprocessed]
        return ids
    def decode(self,ids):
        text=" ".join(self.int_to_str[i] for i in ids)
        text=re.sub(r'\s([,.:;?_!"()\']|--)\s', r'\1', text)
        return text

In [13]:
# testing tokenizer
tokenizer=SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable pride."""
ids=tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [14]:
print(tokenizer.decode(ids))

" It's the last he painted,you know," Mrs.Gisburn said with pardonable pride .


In [15]:
# modifying the tokenizer to handle unknown tokens
all_tokens=sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab={token:int for int,token in enumerate(all_tokens)}

print("vocabulary size:", len(vocab))

vocabulary size: 1132


In [18]:
list(vocab.items())[-5:]

[('younger', 1127),
 ('your', 1128),
 ('yourself', 1129),
 ('<|endoftext|>', 1130),
 ('<|unk|>', 1131)]

In [28]:
# a simple tokenizer that can handle unknown tokens and end of text token
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}
    
    def encode(self,text):
        preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        preprocessed=[item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids=[self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)    #2
        return text
    

In [29]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [30]:
tokenizer=SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))


[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [31]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [32]:
# BPE- Byte Pair Encoding
